# Phase 5 (LSI): one Phase 1 model, everything finished by a set time
Set `MODEL` (and `FINISH_IST`) in cell 2, then **Save Version -> Save & Run All**. One notebook per model
(llama, decoder, moe, encdec). Every model gets: mid-training (not encdec) -> 2 main fine-tuning techniques ->
whole-document stage -> knowledge distillation -> 2 more fine-tuning techniques -> ensemble -> zip.
Fine-tuning techniques (all our own code): LoRA + LoRMA (main), QLoRA + adapter; spares full, BitFit, partial, frozen. Every run
measures its own speed and trains only as much as fits before the finishing time.

In [ ]:
# 1. setup: code from GitHub
import os, subprocess, time

T0 = time.time()
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
try:
    os.environ["HF_TOKEN"] = s.get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: data download without it")
GH = s.get_secret("GH_TOKEN")
REPO = "/kaggle/working/cs781-capstone"
cmds = (
    [["git", "clone", "-q", f"https://{GH}@github.com/NaveenOLA13/cs781-capstone.git", REPO]]
    if not os.path.isdir(REPO)
    else [["git", "-C", REPO, "fetch", "-q"], ["git", "-C", REPO, "reset", "-q", "--hard", "origin/main"]]
)
for c in cmds:
    if subprocess.run(c, capture_output=True).returncode != 0:
        raise RuntimeError("git failed: check the GH_TOKEN secret is attached to this notebook")
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# 2. settings: model, finishing time, plan
MODEL = "llama"  # "llama" | "decoder" | "moe" | "encdec"
FINISH_IST = "22:30"  # everything (results zip included) finished by this time, India time, today
RESUME = None  # "auto": continue an attached earlier version of this notebook

PLAN = {  # mid-training tokens (0 = none); 2 main techniques (whole-document stage + KD teacher come from them);
    # 2 more techniques; spares (used if a technique fails, or if time is left); KD student size (0 = same)
    "llama": dict(mid=20_000_000, batch=8, kd_layers=0.5),
    "decoder": dict(mid=20_000_000, batch=8, kd_layers=0.5),
    "moe": dict(mid=8_000_000, batch=4, kd_layers=0.5),
    "encdec": dict(mid=0, batch=8, kd_layers=0),
}[MODEL]
PLAN.update(main=("lora", "lorma"), more=("qlora", "adapter"), spare=("full", "bitfit", "partial", "frozen"))
COMMON = dict(
    epochs=1,
    train_chunks=3,
    eval_chunks=16,
    doc_tokens=8192,
    batch_size=PLAN["batch"],
    eval_batch_size=16 if MODEL == "moe" else 32,
    log_every=500,
)
OUT = "/kaggle/working/lsi_runs"
ENS = f"/kaggle/working/lsi_ensemble_{MODEL}"

import sys, glob, shutil, json, datetime
from zoneinfo import ZoneInfo

sys.path.insert(0, f"{REPO}/phase5")
IST = ZoneInfo("Asia/Kolkata")
_now = datetime.datetime.now(IST)
_h, _m = map(int, FINISH_IST.split(":"))
FINISH = _now.replace(hour=_h, minute=_m, second=0, microsecond=0).timestamp()
TRAIN_END = FINISH - 15 * 60  # training stops here; the last 15 min: ensemble, table, zip, saving


def minutes_left(t=None):
    return ((TRAIN_END if t is None else t) - time.time()) / 60


def ist(t):
    return datetime.datetime.fromtimestamp(t, IST).strftime("%H:%M")


assert minutes_left() > 90, f"only {minutes_left():.0f} min until {ist(TRAIN_END)} IST: set a later FINISH_IST"
print(
    f"now {_now:%H:%M} IST; training until {ist(TRAIN_END)}, everything finished by {ist(FINISH)} IST "
    f"({minutes_left(FINISH):.0f} min from now)"
)

HINTS = {"llama": ["llama"], "decoder": ["decoder"], "encdec": ["encdec", "encoder_decoder", "enc_dec"], "moe": ["moe"]}


def find_ckpt(model):
    c = set(glob.glob("/kaggle/input/**/*.pt", recursive=True))
    c |= {os.path.dirname(p) for p in glob.glob("/kaggle/input/**/data.pkl", recursive=True)}
    hits = []
    for p in sorted(c):
        low = p.lower().replace("-", "_")
        if "last" in os.path.basename(low) or "/mid5/" in low or "lsi_runs" in low:
            continue
        if model == "decoder" and any(h in low for h in HINTS["encdec"]):
            continue
        if any(h in low for h in HINTS[model]):
            hits.append(p)
    hits.sort(key=lambda p: (not p.endswith(".pt"), len(p)))
    return hits[0] if hits else None


CKPT = find_ckpt(MODEL)
print("Phase 1 checkpoint:", CKPT)
assert CKPT, "attach the checkpoints-models dataset (and moe-checkpoint for MoE) and wait until it is ready"
if RESUME == "auto":
    found = sorted(glob.glob("/kaggle/input/**/lsi_runs", recursive=True))
    RESUME = os.path.dirname(found[0]) if found else None
    print("resuming from", RESUME)
if RESUME:
    for sub in ("lsi_runs", "mid5", "lsi_cache"):
        src = os.path.join(RESUME, sub)
        if os.path.isdir(src) and not os.path.isdir(f"/kaggle/working/{sub}"):
            shutil.copytree(src, f"/kaggle/working/{sub}")
            print("resumed", src)

In [ ]:
# 3. the data: sizes, statutes, what the documents look like
from lsi_data import load_data, summary, explore

data = load_data(log=print)
print(summary(data))
print(explore(data))

In [ ]:
# 4. self-tests on tiny stand-in models (~6 min); stops the notebook if something is broken
if not os.path.exists("/kaggle/working/tests_passed"):
    r = subprocess.run(
        f"cd {REPO}/phase5 && python test_lsi_data.py && python test_lsi_mid.py && "
        f"python test_lsi_train.py && python test_lsi_hier.py",
        shell=True,
        capture_output=True,
        text=True,
    )
    print(r.stdout[-4000:], r.stderr[-3000:])
    assert r.returncode == 0, "self-test failed: send the output above"
    open("/kaggle/working/tests_passed", "w").write("ok")
print(f"tests passed; {minutes_left():.0f} min of training time left")

In [ ]:
# 5. mid-training: next-token prediction on the LSI training + dev text (no labels, no test text)
if PLAN["mid"]:
    from lsi_mid import mid_train

    START = mid_train(MODEL, path=CKPT, tokens=PLAN["mid"], batch_size=8 if MODEL == "moe" else 16)
else:
    START = CKPT  # the encoder-decoder is fine-tuned from its Phase 1 checkpoint
print("fine-tuning starts from", START, f"| {minutes_left():.0f} min of training time left")

In [ ]:
# 6. the 2 main techniques (one per GPU), every chunk's vector kept for the whole-document stage
from lsi_grid import run_grid, run_kd, ensemble, compare_all, report

STAGE_B_MIN = 20
main_end = time.time() + 0.5 * (TRAIN_END - time.time() - STAGE_B_MIN * 60)
print(f"main techniques {PLAN['main']} until {ist(main_end)} IST")
try:
    run_grid(MODEL, techniques=PLAN["main"], ckpt=START, out_dir=OUT, save_chunks=True, finish_by=main_end, **COMMON)
except Exception as e:
    print("main runs failed:", type(e).__name__, e)
print(f"{minutes_left():.0f} min of training time left")

In [ ]:
# 7. whole-document stage: statute-wise attention over all chunks of a document
from lsi_hier import run_stage_b

for tech in PLAN["main"]:
    run_dir = f"{OUT}/{MODEL}_{tech}"
    if os.path.exists(f"{run_dir}/stageB/hier.txt"):
        print(open(f"{run_dir}/stageB/hier.txt").read())
    elif os.path.exists(f"{run_dir}/chunks.pt") and minutes_left() > 40:
        try:
            run_stage_b(run_dir, versions=("gru_attn_res", "gru_attn"), epochs=8, patience=2)
        except Exception as e:
            print("whole-document stage failed for", run_dir, type(e).__name__, e)
print(f"{minutes_left():.0f} min of training time left")

In [ ]:
# 8. knowledge distillation: teacher = the best whole-document (or main) result of this model; student = the same
#    mid-trained model with half its blocks (encdec: same size), with the teacher and, for comparison, without it
cands = []
for tech in PLAN["main"]:
    run_dir = f"{OUT}/{MODEL}_{tech}"
    files = glob.glob(f"{run_dir}/stageB/*/metrics.json")
    if os.path.exists(f"{run_dir}/train_probs.json"):
        files.append(f"{run_dir}/metrics.json")
    for f in files:
        m = json.load(open(f))
        cands.append((max(m["cv_global"], m["cv_per_label"]), os.path.dirname(f)))
KD_DONE = lambda: os.path.exists(f"{OUT}/{MODEL}_student_kd/metrics.json")
if not cands:
    print("no teacher (the main runs did not finish): knowledge distillation skipped")
elif not KD_DONE() and minutes_left() > 30:
    teacher = max(cands)[1]
    kd_end = time.time() + 0.45 * (TRAIN_END - time.time())
    print(f"teacher {teacher} (cross-fitted dev macro-F1 {max(cands)[0]:.4f}); KD until {ist(kd_end)} IST")
    try:
        run_kd(
            teacher,
            student_layers=PLAN["kd_layers"],
            plain=bool(PLAN["kd_layers"]),
            out_dir=OUT,
            finish_by=kd_end,
            **COMMON,
        )
    except Exception as e:
        print("KD failed:", type(e).__name__, e)
    if PLAN["kd_layers"] and not KD_DONE() and minutes_left() > 30:
        print("the half-size student failed (see its log.txt): KD again with a same-size student")
        kd_end = time.time() + 0.45 * (TRAIN_END - time.time())
        try:
            run_kd(teacher, student_layers=0, plain=False, out_dir=OUT, finish_by=kd_end, **COMMON)
        except Exception as e:
            print("KD failed again:", type(e).__name__, e)
print(f"{minutes_left():.0f} min of training time left")

In [ ]:
# 9. 2 more techniques (70% of the time left; spares if one fails, else a bonus pair if 30+ min are left)
ALL = list(PLAN["main"]) + list(PLAN["more"]) + list(PLAN["spare"])
finished = lambda t: os.path.exists(f"{OUT}/{MODEL}_{t}/metrics.json")
queue = [t for t in list(PLAN["more"]) + list(PLAN["spare"]) if not finished(t)]
while queue:
    n_done = sum(finished(t) for t in ALL)
    if minutes_left() < (30 if n_done >= 4 else 25):
        break
    pair, queue = tuple(queue[:2]), queue[2:]
    needed = n_done + len(pair) <= 4 and queue  # still needed for the 4, and spares exist
    end = time.time() + 0.7 * (TRAIN_END - time.time()) if needed else TRAIN_END  # keep time for a spare
    print(f"techniques {pair} until {ist(end)} IST ({n_done} techniques finished so far)")
    try:
        run_grid(MODEL, techniques=pair, ckpt=START, out_dir=OUT, finish_by=end, **COMMON)
    except Exception as e:
        print("runs failed:", type(e).__name__, e)
print(f"finished techniques: {[t for t in ALL if finished(t)]}; KD student: {KD_DONE()}")

In [ ]:
# 10. ensemble of this notebook's results + one table of everything + the list of submission files
try:
    compare_all(OUT)
    ensemble([OUT], out_dir=ENS)
except Exception as e:
    print("ensemble skipped:", type(e).__name__, e)
try:
    report([OUT, ENS] if os.path.isdir(ENS) else [OUT], out_path="/kaggle/working/all_results.txt")
except Exception as e:
    print("report failed:", type(e).__name__, e)
rows = []
for f in glob.glob(f"{OUT}/**/metrics.json", recursive=True) + glob.glob(f"{ENS}/metrics.json"):
    m = json.load(open(f))
    sub = os.path.join(os.path.dirname(f), "submission", "submission.zip")
    if "dev_chosen" in m and os.path.exists(sub):
        rows.append((max(m["cv_global"], m["cv_per_label"]), os.path.relpath(sub, "/kaggle/working")))
lines = [f"{cv:.4f}  {p}" for cv, p in sorted(rows, reverse=True)]
print("\nsubmission files, best cross-fitted dev macro-F1 first:\n" + "\n".join(lines))
open("/kaggle/working/submission_files.txt", "w").write("\n".join(lines) + "\n")

In [ ]:
# 11. results zip (every submission, log, score; no weights) + delete big temporary files so saving is quick
import zipfile
from IPython.display import FileLink, display

name = f"lsi_results_{MODEL}.zip"
with zipfile.ZipFile(f"/kaggle/working/{name}", "w", zipfile.ZIP_DEFLATED) as z:
    for base in (OUT, ENS, "/kaggle/working/mid5"):
        for root, _, files in os.walk(base):
            for f in files:
                if not f.endswith(".pt"):
                    full = os.path.join(root, f)
                    z.write(full, os.path.relpath(full, "/kaggle/working"))
    for f in ("all_results.txt", "submission_files.txt"):
        if os.path.exists(f"/kaggle/working/{f}"):
            z.write(f"/kaggle/working/{f}", f)
for p in glob.glob(f"{OUT}/*/chunks.pt"):
    os.remove(p)
shutil.rmtree("/kaggle/working/lsi_cache", ignore_errors=True)
display(FileLink(name))
print(f"done at {ist(time.time())} IST")